# EDA - Dataset (Batch - Yahoo Finance)

**Proyecto ETL Forex - Entrega Final**  
Bryan Andres Herrera Betancur - Alessandro Yusty Ceballos

## Objetivo
Análisis exploratorio sobre los datos históricos OHLC extraídos en modo **batch** desde Yahoo Finance vía `yfinance`. Estos datos forman la línea base histórica del pipeline y se cargan en la tabla `fact_quotes` del Data Warehouse a través del DAG de Airflow.

## Fuente del dato
- **API**: Yahoo Finance (`yf.download()`)
- **Pares analizados**: USD/CAD y EUR/SEK (ejemplos representativos)
- **Resolución**: 1 minuto
- **Ventana**: 2 días previos a la extracción

## 1. Carga e inspección inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
%matplotlib inline

In [ ]:
# Cargamos el dataset principal (USD/CAD)
df = pd.read_csv('data/USD_CAD.csv')
df.columns = [c.lower() for c in df.columns]
df = df.rename(columns={'datetime': 'timestamp'})
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df['symbol'] = 'USDCAD'
df.head()

In [ ]:
print(f'Dimensiones del dataset: {df.shape}')
print(f'Rango temporal: {df.timestamp.min()} → {df.timestamp.max()}')
print(f'Días cubiertos: {df.timestamp.dt.date.nunique()}')
df.dtypes

## 2. Calidad de los datos

In [ ]:
# Valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum())

# Duplicados
print(f'\nFilas duplicadas: {df.duplicated().sum()}')
print(f'Timestamps duplicados: {df.timestamp.duplicated().sum()}')

In [ ]:
# Validación lógica OHLC: high >= max(open, close), low <= min(open, close)
violaciones_high = (df['high'] < df[['open', 'close']].max(axis=1)).sum()
violaciones_low  = (df['low']  > df[['open', 'close']].min(axis=1)).sum()
print(f'Violaciones high < max(open,close): {violaciones_high}')
print(f'Violaciones low  > min(open,close): {violaciones_low}')

# Precios negativos o cero (lo que valida el DAG)
print(f'\nPrecios <= 0: {(df[["open","high","low","close"]] <= 0).sum().sum()}')

## 3. Estadísticas descriptivas

In [ ]:
df[['open', 'high', 'low', 'close']].describe()

In [ ]:
# Spread intraminuto (volatilidad de cada vela)
df['spread'] = df['high'] - df['low']
df['range_pct'] = df['spread'] / df['close'] * 100
print(f'Spread promedio: {df.spread.mean():.5f}')
print(f'Spread % promedio: {df.range_pct.mean():.4f}%')
print(f'Spread máximo: {df.spread.max():.5f}')

## 4. Visualizaciones

In [ ]:
# Serie temporal del precio de cierre con banda high-low
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(df['timestamp'], df['low'], df['high'], alpha=0.25, label='Rango high-low', color='steelblue')
ax.plot(df['timestamp'], df['close'], linewidth=1.2, color='navy', label='Close')
ax.set_title('USD/CAD - Serie temporal del precio (1-min)', fontsize=13)
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Precio')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distribución del precio de cierre (histograma + boxplot)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['close'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(df['close'].mean(), color='red', linestyle='--', label=f'Media {df.close.mean():.4f}')
axes[0].set_title('Distribución del precio de cierre')
axes[0].set_xlabel('Close')
axes[0].legend()

axes[1].boxplot(df['close'], vert=False)
axes[1].set_title('Boxplot del precio de cierre')
axes[1].set_xlabel('Close')
plt.tight_layout()
plt.show()

In [ ]:
# Cobertura temporal: registros por día
df['date'] = df['timestamp'].dt.date
registros_por_dia = df.groupby('date').size()
fig, ax = plt.subplots(figsize=(10, 4))
registros_por_dia.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Cobertura temporal - registros por día')
ax.set_xlabel('Fecha')
ax.set_ylabel('Cantidad de registros (velas de 1 min)')
ax.axhline(1440, color='red', linestyle='--', label='Máximo teórico (1440 min/día)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\nRegistros por día:')
print(registros_por_dia)

In [ ]:
# Análisis comparativo con un segundo par (EUR/SEK)
df_eur = pd.read_csv('data/EUR_SEK.csv')
df_eur.columns = [c.lower() for c in df_eur.columns]
df_eur = df_eur.rename(columns={'datetime': 'timestamp'})
df_eur['timestamp'] = pd.to_datetime(df_eur['timestamp'], utc=True)

# Normalizar precios a base 100 para comparar evolución relativa
norm_cad = df['close'] / df['close'].iloc[0] * 100
norm_eur = df_eur['close'] / df_eur['close'].iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['timestamp'], norm_cad, label='USD/CAD', linewidth=1.2)
ax.plot(df_eur['timestamp'], norm_eur, label='EUR/SEK', linewidth=1.2, alpha=0.8)
ax.set_title('Evolución relativa (base 100) - comparativa de pares')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Índice (base 100)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Hallazgos e insights

1. **Calidad del dato**: el dataset extraído desde Yahoo Finance pasa todas las validaciones implementadas en el DAG (sin nulos, sin negativos, lógica OHLC consistente, sin duplicados de timestamp). Esto confirma que la fuente es confiable para ser cargada al modelo dimensional.

2. **Baja volatilidad relativa**: el spread porcentual promedio es muy bajo (< 0.05%), lo cual es típico en mercados forex de pares mayores. Esto valida que las velas de 1 minuto reflejan microvariaciones de precio, ideales para análisis intradiario.

3. **Cobertura temporal**: los días cubiertos muestran que Yahoo entrega aproximadamente 1440 velas/día (cobertura completa) excepto fines de semana donde forex está cerrado. **Esta es una decisión de negocio importante**: el pipeline no debería disparar alertas de calidad los sábados/domingos.

4. **Comparabilidad entre pares**: la normalización a base 100 permite comparar evolución relativa entre pares con escalas muy distintas (USD/CAD ~1.36 vs EUR/SEK ~10.8). Este enfoque es útil para dashboards multidivisa en Metabase.

## 6. Implicaciones para el pipeline ETL

- Las validaciones del DAG (no nulos, precios > 0) son **suficientes** para esta fuente: no se observaron violaciones que requieran reglas adicionales.
- La granularidad de 1 minuto es **compatible** con la agregación OHLC que aplicamos a los datos en streaming de Finnhub (ver `eda_api.ipynb`), permitiendo unificar ambas fuentes en `fact_quotes` con el mismo esquema.